# Week 12: LLMs, bounded agents, and vibe coding - Exercise

Machine Learning in Engineering Science | NCKU + LAiMM | Fall 2026

**Meeting:** Monday 2026-11-23, 14:00-17:00. Three one-hour modules; 60 minutes explanation and 120 minutes practice across slides and this notebook.

**Learning outcome:** Explain next-token prediction; validate structured tool proposals; retain control outside the model.

**Scope:** Train a transparent character bigram language model, then inspect a separate deterministic proposal validator. Neither the bigram nor the rule dispatcher is a Transformer or a live LLM agent.

**Free path:** use Python 3, a CPU Colab runtime or Jupyter. No API key, GPU, paid inference, confidential data or external dataset download is required. First-time package installation needs internet. Record actual versions; do not assume a Colab version.

**How to work:** read a task; edit its prediction field before running; run its code with Shift+Enter; inspect the output; write an evidence-based explanation. Core scaffolds run as supplied so you can start without filling a programming puzzle. Make only the requested controlled change. Use an AI coding assistant only if desired and disclose what you checked.

**Practice policy:** 8 core tasks and 2 optional challenges. In-class practice, not additional weekly graded homework. Weeks 11 and 18 remain written and oral examinations.

**Recovery:** if a variable is missing, restart and run all from the top. Initialization resets responses; after editing answers, rerun only the final export cell. Save your own edited notebook as well as the ZIP; the exported analysis source is the supplied scaffold snapshot, not a capture of unsaved code edits.

## 0. Environment and response record

Run this once at the beginning. It installs only missing free packages, creates a week-specific artifact directory, and initializes 19 blank reflection fields. Printouts are observations, not answer keys.

In [ ]:
WEEK=12
WORKFLOW_TEXT=''
RUNNER_TEXT=''
ANALYSIS_SOURCE='WEEK=12\nWORKFLOW_TEXT=\'\'\nRUNNER_TEXT=\'\'\n\nimport sys, subprocess, importlib.util, platform, json, math, hashlib, statistics\nimport tempfile, zipfile, shutil, copy, random\nfrom pathlib import Path\nmissing=[p for m,p in [(\'numpy\',\'numpy\'),(\'matplotlib\',\'matplotlib\'),(\'sklearn\',\'scikit-learn\')] if importlib.util.find_spec(m) is None]\nif missing:\n    subprocess.check_call([sys.executable,\'-m\',\'pip\',\'install\',*missing])\nimport numpy as np\nimport matplotlib\nimport matplotlib.pyplot as plt\nimport sklearn\nSEED=2026\nrng=np.random.default_rng(SEED)\nrandom.seed(SEED)\nARTIFACTS=Path(\'artifacts\')/f\'week{WEEK}\'\nARTIFACTS.mkdir(parents=True,exist_ok=True)\nENV={\'python\':platform.python_version(),\'numpy\':np.__version__,\'matplotlib\':matplotlib.__version__,\'scikit_learn\':sklearn.__version__,\'platform\':platform.platform()}\ndef canonical(x): return json.dumps(x,sort_keys=True,allow_nan=False,separators=(\',\',\':\')).encode()\ndef digest(x): return hashlib.sha256(canonical(x)).hexdigest()\ndef save_json(name,obj):\n    p=ARTIFACTS/name; p.write_text(json.dumps(obj,indent=2,allow_nan=False)); return p\ndef plot(name,labels,values,ylabel,title):\n    fig,ax=plt.subplots(figsize=(9,4.8),layout=\'constrained\')\n    ax.bar(labels,values,color=[\'#167b87\',\'#7f2635\',\'#b88a29\',\'#12233f\']*10)\n    ax.set_ylabel(ylabel); ax.set_title(title)\n    ax.tick_params(axis=\'x\',labelrotation=18); ax.spines[[\'top\',\'right\']].set_visible(False)\n    fig.savefig(ARTIFACTS/name,dpi=160); plt.show(); plt.close(fig)\nobservations={}\nresponses={**{f\'task{i}_{p}\':\'\' for i in range(1,9) for p in [\'prediction\',\'explanation\']},\'controlled_change\':\'\',\'limitation\':\'\',\'ai_disclosure\':\'\'}\nprint(\'Environment:\',ENV,\'; seed\',SEED)\n\n\nCORPUS=(\'measure heat and record units. review the source. compute a mean. \'\n        \'reject unknown tools. explain the evidence. preserve the inputs. \')*5\nvocab=sorted(set(CORPUS)); stoi={c:i for i,c in enumerate(vocab)}\ncounts=np.zeros((len(vocab),len(vocab)),dtype=np.int64)\nfor a,b in zip(CORPUS[:-1],CORPUS[1:]): counts[stoi[a],stoi[b]]+=1\nprobs=(counts+1)/(counts.sum(axis=1,keepdims=True)+len(vocab))\nassert np.allclose(probs.sum(axis=1),1)\ndef softmax_temperature(logits,t):\n    if not math.isfinite(t) or t<=0: raise ValueError(\'positive finite temperature required\')\n    z=np.asarray(logits,dtype=float)/t; p=np.exp(z-z.max()); return p/p.sum()\ndef generate(t,seed=SEED,n=100):\n    local=np.random.default_rng(seed); out=[\' \']\n    for _ in range(n): out.append(local.choice(vocab,p=softmax_temperature(np.log(probs[stoi[out[-1]]]),t)))\n    return \'\'.join(out)\ndef parse_proposal(raw):\n    try:\n        def no_constant(s): raise ValueError(\'nonstandard numeric constant \'+s)\n        def no_duplicate(pairs):\n            out={}\n            for k,v in pairs:\n                if k in out: raise ValueError(\'duplicate field \'+k)\n                out[k]=v\n            return out\n        return {\'status\':\'parsed\',\'proposal\':json.loads(raw,parse_constant=no_constant,object_pairs_hook=no_duplicate)}\n    except (ValueError,TypeError): return {\'status\':\'parse_error\',\'proposal\':None}\ndef dispatch(p,allow=(\'mean\',)):\n    if type(p) is not dict or set(p)!={\'tool\',\'values\'}: return {\'status\':\'invalid_schema\',\'result\':None}\n    if type(p[\'tool\']) is not str or p[\'tool\'] not in allow: return {\'status\':\'forbidden_tool\',\'result\':None}\n    a=p[\'values\']\n    if type(a) is not list or not 1<=len(a)<=100 or not all(type(v) in (int,float) and abs(v)<=1e100 and math.isfinite(v) for v in a):\n        return {\'status\':\'invalid_data\',\'result\':None}\n    value=math.fsum(v/len(a) for v in a) if p[\'tool\']==\'mean\' else statistics.median(a)\n    if not math.isfinite(value): return {\'status\':\'nonfinite_result\',\'result\':None}\n    return {\'status\':\'ok\',\'result\':value}\ndef bounded(proposals,cap=2):\n    trace=[]\n    for i,p in enumerate(proposals):\n        if i>=cap:\n            trace.append({\'attempt\':i+1,\'status\':\'budget_exhausted\'}); break\n        trace.append(dict(attempt=i+1,proposal=p,**dispatch(p)))\n    return trace\nprint(\'Original teaching corpus SHA256:\',hashlib.sha256(CORPUS.encode()).hexdigest())\n\nrow=stoi[\' \']; order=np.argsort(-probs[row]); top=order[:6]\nobservations[\'task1\']={\'context\':\'space\',\'top\':[{\'next\':vocab[i],\'count\':int(counts[row,i]),\'probability\':float(probs[row,i])} for i in top]}\nassert np.isclose(probs[row].sum(),1)\nplot(\'distribution.png\',[repr(vocab[i]) for i in top],probs[row,top],\'Probability\',\'Trained character bigram: successors of space\')\nprint(observations[\'task1\'])\nlogits=np.log(probs[stoi[\' \']])\nentropies={str(t):float(-np.sum((p:=softmax_temperature(logits,t))*np.log(p))) for t in [0.5,1.,2.]}\nobservations[\'task2\']={\'entropy_nats\':entropies,\'samples\':{str(t):generate(t) for t in [0.5,2.]},\'fixture_losses\':[-math.log(.8),-math.log(.2)]}\nassert entropies[\'0.5\']<=entropies[\'2.0\']\nplot(\'temperature.png\',list(entropies),list(entropies.values()),\'Entropy (nats)\',\'Same bigram logits; different sampling temperatures\')\nprint(observations[\'task2\'])\nproposals=[{\'tool\':\'mean\',\'values\':[1,2,6]},{\'tool\':\'delete_files\',\'values\':[1]}]\noutcomes=[dispatch(p) for p in proposals]\nassert outcomes==[{\'status\':\'ok\',\'result\':3.0},{\'status\':\'forbidden_tool\',\'result\':None}]\nobservations[\'task3\']={\'proposals\':proposals,\'outcomes\':outcomes}; print(outcomes)\nbad=[[],[True],[\'ignore rules\'],[float(\'nan\')],[float(\'inf\')],[1]*101]\nstatuses=[dispatch({\'tool\':\'mean\',\'values\':a})[\'status\'] for a in bad]\nlarge=dispatch({\'tool\':\'mean\',\'values\':[1e100,1e100]})\nassert statuses==[\'invalid_data\']*len(bad) and math.isfinite(large[\'result\'])\nobservations[\'task4\']={\'case_names\':[\'empty\',\'boolean\',\'string\',\'NaN\',\'infinite\',\'too_many\'],\'statuses\':statuses,\'large_finite_result\':large}\nprint(observations[\'task4\'])\ntrace=bounded([{\'tool\':\'unknown\',\'values\':[1]}]*3,cap=2)\nassert len(trace)==3 and trace[-1][\'status\']==\'budget_exhausted\'\nobservations[\'task5\']={\'cap\':2,\'trace\':trace}; print(trace)\nraw=[\'{"tool":"mean","values":[1,2,6]}\',\'{"tool":"mean","values":[1,2,6],}\',\n     \'{"tool":"mean","tool":"median","values":[1]}\',\'{"tool":"mean","values":[NaN]}\']\nparsed=[parse_proposal(s) for s in raw]\nrepaired=raw[1].replace(\',}\',\'}\')\nassert [p[\'status\'] for p in parsed]==[\'parsed\',\'parse_error\',\'parse_error\',\'parse_error\']\nchecked=dispatch(parse_proposal(repaired)[\'proposal\']); assert checked[\'result\']==3\nobservations[\'task6\']={\'raw_strings\':raw,\'parse_statuses\':[p[\'status\'] for p in parsed],\'repaired\':repaired,\'checked\':checked}\nprint(observations[\'task6\'])\nfixture={\'values\':[1,2,6],\'expected_mean\':3.0}\nactual=dispatch({\'tool\':\'mean\',\'values\':fixture[\'values\']})\nwrong=sum(fixture[\'values\'])/(len(fixture[\'values\'])+1)\nassert math.isclose(actual[\'result\'],fixture[\'expected_mean\'],abs_tol=1e-12)\nobservations[\'task7\']={\'fixture\':fixture,\'actual\':actual,\'wrong_denominator_result\':wrong}; print(observations[\'task7\'])\nobservations[\'task8\']={\'corpus_sha256\':hashlib.sha256(CORPUS.encode()).hexdigest(),\'vocabulary\':vocab,\n                       \'scope\':\'Trained character bigram plus deterministic tool-boundary simulation; no live LLM evaluation.\'}\nassert set(observations)=={f\'task{i}\' for i in range(1,9)}\nprint(observations[\'task8\'])\nRUN_CHALLENGES=False\n\noptional={}\nif RUN_CHALLENGES:\n    odd=dispatch({\'tool\':\'median\',\'values\':[1,7,2]},allow=(\'mean\',\'median\'))\n    even=dispatch({\'tool\':\'median\',\'values\':[1,7,2,8]},allow=(\'mean\',\'median\'))\n    empty=dispatch({\'tool\':\'median\',\'values\':[]},allow=(\'mean\',\'median\'))\n    assert odd[\'result\']==2 and even[\'result\']==4.5 and empty[\'status\']==\'invalid_data\'\n    optional[\'A_median\']={\'odd\':odd,\'even\':even,\'empty\':empty}\n    variants=[\' {"values":[1,2,6],"tool":"mean"} \',\'{"tool":"mean","values":[1,2,6]}\']\n    checked=[dispatch(parse_proposal(v)[\'proposal\']) for v in variants]\n    assert checked[0]==checked[1]\n    optional[\'B_formatting\']={\'outcomes\':checked,\'limitation\':\'Only equivalent JSON formatting tested, not semantic paraphrase understanding.\'}\nprint(optional)\n'

import sys, subprocess, importlib.util, platform, json, math, hashlib, statistics
import tempfile, zipfile, shutil, copy, random
from pathlib import Path
missing=[p for m,p in [('numpy','numpy'),('matplotlib','matplotlib'),('sklearn','scikit-learn')] if importlib.util.find_spec(m) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install',*missing])
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import sklearn
SEED=2026
rng=np.random.default_rng(SEED)
random.seed(SEED)
ARTIFACTS=Path('artifacts')/f'week{WEEK}'
ARTIFACTS.mkdir(parents=True,exist_ok=True)
ENV={'python':platform.python_version(),'numpy':np.__version__,'matplotlib':matplotlib.__version__,'scikit_learn':sklearn.__version__,'platform':platform.platform()}
def canonical(x): return json.dumps(x,sort_keys=True,allow_nan=False,separators=(',',':')).encode()
def digest(x): return hashlib.sha256(canonical(x)).hexdigest()
def save_json(name,obj):
    p=ARTIFACTS/name; p.write_text(json.dumps(obj,indent=2,allow_nan=False)); return p
def plot(name,labels,values,ylabel,title):
    fig,ax=plt.subplots(figsize=(9,4.8),layout='constrained')
    ax.bar(labels,values,color=['#167b87','#7f2635','#b88a29','#12233f']*10)
    ax.set_ylabel(ylabel); ax.set_title(title)
    ax.tick_params(axis='x',labelrotation=18); ax.spines[['top','right']].set_visible(False)
    fig.savefig(ARTIFACTS/name,dpi=160); plt.show(); plt.close(fig)
observations={}
responses={**{f'task{i}_{p}':'' for i in range(1,9) for p in ['prediction','explanation']},'controlled_change':'','limitation':'','ai_disclosure':''}
print('Environment:',ENV,'; seed',SEED)

## 1. Inspect the reference mechanism

Train a transparent character bigram language model, then inspect a separate deterministic proposal validator. Neither the bigram nor the rule dispatcher is a Transformer or a live LLM agent.

Read the definitions below. Identify the data boundary, the allowed operations and a condition that should stop execution. The executable reference is deliberately small enough to inspect.

In [ ]:
CORPUS=('measure heat and record units. review the source. compute a mean. '
        'reject unknown tools. explain the evidence. preserve the inputs. ')*5
vocab=sorted(set(CORPUS)); stoi={c:i for i,c in enumerate(vocab)}
counts=np.zeros((len(vocab),len(vocab)),dtype=np.int64)
for a,b in zip(CORPUS[:-1],CORPUS[1:]): counts[stoi[a],stoi[b]]+=1
probs=(counts+1)/(counts.sum(axis=1,keepdims=True)+len(vocab))
assert np.allclose(probs.sum(axis=1),1)
def softmax_temperature(logits,t):
    if not math.isfinite(t) or t<=0: raise ValueError('positive finite temperature required')
    z=np.asarray(logits,dtype=float)/t; p=np.exp(z-z.max()); return p/p.sum()
def generate(t,seed=SEED,n=100):
    local=np.random.default_rng(seed); out=[' ']
    for _ in range(n): out.append(local.choice(vocab,p=softmax_temperature(np.log(probs[stoi[out[-1]]]),t)))
    return ''.join(out)
def parse_proposal(raw):
    try:
        def no_constant(s): raise ValueError('nonstandard numeric constant '+s)
        def no_duplicate(pairs):
            out={}
            for k,v in pairs:
                if k in out: raise ValueError('duplicate field '+k)
                out[k]=v
            return out
        return {'status':'parsed','proposal':json.loads(raw,parse_constant=no_constant,object_pairs_hook=no_duplicate)}
    except (ValueError,TypeError): return {'status':'parse_error','proposal':None}
def dispatch(p,allow=('mean',)):
    if type(p) is not dict or set(p)!={'tool','values'}: return {'status':'invalid_schema','result':None}
    if type(p['tool']) is not str or p['tool'] not in allow: return {'status':'forbidden_tool','result':None}
    a=p['values']
    if type(a) is not list or not 1<=len(a)<=100 or not all(type(v) in (int,float) and abs(v)<=1e100 and math.isfinite(v) for v in a):
        return {'status':'invalid_data','result':None}
    value=math.fsum(v/len(a) for v in a) if p['tool']=='mean' else statistics.median(a)
    if not math.isfinite(value): return {'status':'nonfinite_result','result':None}
    return {'status':'ok','result':value}
def bounded(proposals,cap=2):
    trace=[]
    for i,p in enumerate(proposals):
        if i>=cap:
            trace.append({'attempt':i+1,'status':'budget_exhausted'}); break
        trace.append(dict(attempt=i+1,proposal=p,**dispatch(p)))
    return trace
print('Original teaching corpus SHA256:',hashlib.sha256(CORPUS.encode()).hexdigest())

## Task 1. Inspect a learned next-character distribution

Predict the most frequent successor of a space. Read the actual transition counts, then compare the normalized add-one probabilities.

1. Predict the outcome in the response cell below before running the analysis.
2. Run the next analysis cell. Read all printed fields, not just the last number.
3. Explain the mechanism using an observed value, case ID, rejection or source.
4. If an assertion fails unexpectedly, preserve the error and inspect the relevant contract; do not delete the assertion.

**Self-check:** All successor probabilities, not just the six plotted, sum to one. A smoothed nonzero probability does not imply an observed transition.

In [ ]:
responses['task1_prediction']=''  # Write before running the next cell.
responses['task1_explanation']=''  # Edit after inspecting actual evidence.

In [ ]:
row=stoi[' ']; order=np.argsort(-probs[row]); top=order[:6]
observations['task1']={'context':'space','top':[{'next':vocab[i],'count':int(counts[row,i]),'probability':float(probs[row,i])} for i in top]}
assert np.isclose(probs[row].sum(),1)
plot('distribution.png',[repr(vocab[i]) for i in top],probs[row,top],'Probability','Trained character bigram: successors of space')
print(observations['task1'])

## Task 2. Change temperature without changing the model

Predict how T=0.5 and T=2 change entropy. The corpus and trained counts stay fixed; compare distributions as well as generated samples.

1. Predict the outcome in the response cell below before running the analysis.
2. Run the next analysis cell. Read all printed fields, not just the last number.
3. Explain the mechanism using an observed value, case ID, rejection or source.
4. If an assertion fails unexpectedly, preserve the error and inspect the relevant contract; do not delete the assertion.

**Self-check:** T must be finite and positive. Entropy concentration is not a factual-accuracy measure.

In [ ]:
responses['task2_prediction']=''  # Write before running the next cell.
responses['task2_explanation']=''  # Edit after inspecting actual evidence.

In [ ]:
logits=np.log(probs[stoi[' ']])
entropies={str(t):float(-np.sum((p:=softmax_temperature(logits,t))*np.log(p))) for t in [0.5,1.,2.]}
observations['task2']={'entropy_nats':entropies,'samples':{str(t):generate(t) for t in [0.5,2.]},'fixture_losses':[-math.log(.8),-math.log(.2)]}
assert entropies['0.5']<=entropies['2.0']
plot('temperature.png',list(entropies),list(entropies.values()),'Entropy (nats)','Same bigram logits; different sampling temperatures')
print(observations['task2'])

## Task 3. Reject an unknown tool before execution

Predict statuses for a mean proposal and a forbidden capability. No files, network requests or real messages are touched.

1. Predict the outcome in the response cell below before running the analysis.
2. Run the next analysis cell. Read all printed fields, not just the last number.
3. Explain the mechanism using an observed value, case ID, rejection or source.
4. If an assertion fails unexpectedly, preserve the error and inspect the relevant contract; do not delete the assertion.

**Self-check:** The forbidden proposal produces no numeric result. No eval, shell tool or dynamic import is used to dispatch a name.

In [ ]:
responses['task3_prediction']=''  # Write before running the next cell.
responses['task3_explanation']=''  # Edit after inspecting actual evidence.

In [ ]:
proposals=[{'tool':'mean','values':[1,2,6]},{'tool':'delete_files','values':[1]}]
outcomes=[dispatch(p) for p in proposals]
assert outcomes==[{'status':'ok','result':3.0},{'status':'forbidden_tool','result':None}]
observations['task3']={'proposals':proposals,'outcomes':outcomes}; print(outcomes)

## Task 4. Probe malformed numeric inputs

Predict a reason for each rejection before running. Python values and text JSON use different entrance gates.

1. Predict the outcome in the response cell below before running the analysis.
2. Run the next analysis cell. Read all printed fields, not just the last number.
3. Explain the mechanism using an observed value, case ID, rejection or source.
4. If an assertion fails unexpectedly, preserve the error and inspect the relevant contract; do not delete the assertion.

**Self-check:** All six malformed native inputs reject. Nonstandard JSON NaN is also rejected by the parser in Task 6.

In [ ]:
responses['task4_prediction']=''  # Write before running the next cell.
responses['task4_explanation']=''  # Edit after inspecting actual evidence.

In [ ]:
bad=[[],[True],['ignore rules'],[float('nan')],[float('inf')],[1]*101]
statuses=[dispatch({'tool':'mean','values':a})['status'] for a in bad]
large=dispatch({'tool':'mean','values':[1e100,1e100]})
assert statuses==['invalid_data']*len(bad) and math.isfinite(large['result'])
observations['task4']={'case_names':['empty','boolean','string','NaN','infinite','too_many'],'statuses':statuses,'large_finite_result':large}
print(observations['task4'])

## Task 5. Exhaust the attempt budget

Predict the third status when two proposals have already consumed the two-attempt cap. Rejected proposals consume attempts too.

1. Predict the outcome in the response cell below before running the analysis.
2. Run the next analysis cell. Read all printed fields, not just the last number.
3. Explain the mechanism using an observed value, case ID, rejection or source.
4. If an assertion fails unexpectedly, preserve the error and inspect the relevant contract; do not delete the assertion.

**Self-check:** Only two proposals enter dispatch. The third is a stop event, not a third tool attempt.

In [ ]:
responses['task5_prediction']=''  # Write before running the next cell.
responses['task5_explanation']=''  # Edit after inspecting actual evidence.

In [ ]:
trace=bounded([{'tool':'unknown','values':[1]}]*3,cap=2)
assert len(trace)==3 and trace[-1]['status']=='budget_exhausted'
observations['task5']={'cap':2,'trace':trace}; print(trace)

## Task 6. Repair syntax while preserving the boundary

Compare a valid proposal, a trailing comma, a duplicate key and NaN text. Repair only the syntax of a copied string.

1. Predict the outcome in the response cell below before running the analysis.
2. Run the next analysis cell. Read all printed fields, not just the last number.
3. Explain the mechanism using an observed value, case ID, rejection or source.
4. If an assertion fails unexpectedly, preserve the error and inspect the relevant contract; do not delete the assertion.

**Self-check:** Duplicate fields are not silently overwritten. Syntax repair still passes through the same validator.

In [ ]:
responses['task6_prediction']=''  # Write before running the next cell.
responses['task6_explanation']=''  # Edit after inspecting actual evidence.

In [ ]:
raw=['{"tool":"mean","values":[1,2,6]}','{"tool":"mean","values":[1,2,6],}',
     '{"tool":"mean","tool":"median","values":[1]}','{"tool":"mean","values":[NaN]}']
parsed=[parse_proposal(s) for s in raw]
repaired=raw[1].replace(',}','}')
assert [p['status'] for p in parsed]==['parsed','parse_error','parse_error','parse_error']
checked=dispatch(parse_proposal(repaired)['proposal']); assert checked['result']==3
observations['task6']={'raw_strings':raw,'parse_statuses':[p['status'] for p in parsed],'repaired':repaired,'checked':checked}
print(observations['task6'])

## Task 7. Independently check arithmetic

Calculate 1+2+6 by hand, divide by 3, then check the tool. Explain why zero-mean inputs might hide a denominator bug.

1. Predict the outcome in the response cell below before running the analysis.
2. Run the next analysis cell. Read all printed fields, not just the last number.
3. Explain the mechanism using an observed value, case ID, rejection or source.
4. If an assertion fails unexpectedly, preserve the error and inspect the relevant contract; do not delete the assertion.

**Self-check:** The expected 3.0 was written independently, not generated by dispatch. One fixture is not a full arithmetic evaluation.

In [ ]:
responses['task7_prediction']=''  # Write before running the next cell.
responses['task7_explanation']=''  # Edit after inspecting actual evidence.

In [ ]:
fixture={'values':[1,2,6],'expected_mean':3.0}
actual=dispatch({'tool':'mean','values':fixture['values']})
wrong=sum(fixture['values'])/(len(fixture['values'])+1)
assert math.isclose(actual['result'],fixture['expected_mean'],abs_tol=1e-12)
observations['task7']={'fixture':fixture,'actual':actual,'wrong_denominator_result':wrong}; print(observations['task7'])

## Task 8. Audit what the exported record can support

Check every task is present, preserve a corpus hash and inspect one rejection with its original proposal.

1. Predict the outcome in the response cell below before running the analysis.
2. Run the next analysis cell. Read all printed fields, not just the last number.
3. Explain the mechanism using an observed value, case ID, rejection or source.
4. If an assertion fails unexpectedly, preserve the error and inspect the relevant contract; do not delete the assertion.

**Self-check:** The final export records incomplete reflections as draft. No model account or paid quota is needed.

In [ ]:
responses['task8_prediction']=''  # Write before running the next cell.
responses['task8_explanation']=''  # Edit after inspecting actual evidence.

In [ ]:
observations['task8']={'corpus_sha256':hashlib.sha256(CORPUS.encode()).hexdigest(),'vocabulary':vocab,
                       'scope':'Trained character bigram plus deterministic tool-boundary simulation; no live LLM evaluation.'}
assert set(observations)=={f'task{i}' for i in range(1,9)}
print(observations['task8'])

## Optional challenges

A: add median through a separate allowlist and check odd/even/empty fixtures. B: compare equivalent JSON formatting without granting new capabilities.

Predict first. Set `RUN_CHALLENGES=True` only if core tasks are complete. Record changes separately; these do not change the core completion count.

In [ ]:
RUN_CHALLENGES=False

optional={}
if RUN_CHALLENGES:
    odd=dispatch({'tool':'median','values':[1,7,2]},allow=('mean','median'))
    even=dispatch({'tool':'median','values':[1,7,2,8]},allow=('mean','median'))
    empty=dispatch({'tool':'median','values':[]},allow=('mean','median'))
    assert odd['result']==2 and even['result']==4.5 and empty['status']=='invalid_data'
    optional['A_median']={'odd':odd,'even':even,'empty':empty}
    variants=[' {"values":[1,2,6],"tool":"mean"} ','{"tool":"mean","values":[1,2,6]}']
    checked=[dispatch(parse_proposal(v)['proposal']) for v in variants]
    assert checked[0]==checked[1]
    optional['B_formatting']={'outcomes':checked,'limitation':'Only equivalent JSON formatting tested, not semantic paraphrase understanding.'}
print(optional)

## Final reflection and export

Fill the three reflections below. Keep one limitation even if every assertion passes. The export creates a draft while any of the 19 response fields are blank; ready for peer discussion checks completeness, not correctness. Download the ZIP via the link or the Colab Files sidebar before leaving the temporary runtime.

In [ ]:
responses['controlled_change']=''
responses['limitation']=''
responses['ai_disclosure']=''  # Write none if no AI assistance was used.

In [ ]:
(ARTIFACTS/'supplied-analysis-source.py').write_text(ANALYSIS_SOURCE)

# Edit response assignments above, then rerun this export cell only.
missing_responses=[k for k,v in responses.items() if type(v) is not str or not v.strip()]
record={'week':WEEK,'environment':ENV,'seed':SEED,'observations':observations,'optional':optional,
        'responses':responses,'missing_responses':missing_responses,
        'status':'draft' if missing_responses else 'ready for peer discussion',
        'completion_is_not_correctness':True,'policy':'In-class practice; no new weekly graded homework.'}
save_json('evidence.json',record)
# Snapshot source is supplied by the notebook generator, never invented from output.
(ARTIFACTS/'workflow-source.py').write_text(WORKFLOW_TEXT if WORKFLOW_TEXT else 'No shared thermal workflow in this unit. Inspect the complete notebook source.\n')
payloads=[p for p in ARTIFACTS.rglob('*') if p.is_file() and p not in [ARTIFACTS/'manifest.json',ARTIFACTS/'exercise-evidence.zip'] and not p.name.endswith('.ipynb')]
manifest={str(p.relative_to(ARTIFACTS)):hashlib.sha256(p.read_bytes()).hexdigest() for p in payloads}
save_json('manifest.json',manifest)
archive=ARTIFACTS/'exercise-evidence.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in payloads+[ARTIFACTS/'manifest.json']: z.write(p,arcname=str(p.relative_to(ARTIFACTS)))
with zipfile.ZipFile(archive) as z: assert z.testzip() is None
print('Status:',record['status'],'; missing fields:',missing_responses)
print('Saved:',archive.resolve())
try:
    from IPython.display import display,FileLink
    display(FileLink(str(archive)))
except ImportError:
    pass
# Colab: open the Files sidebar, locate artifacts/weekXX/exercise-evidence.zip,
# use its menu to Download. Download before leaving the temporary runtime.

## Primary references and source boundaries

- [Attention Is All You Need](https://arxiv.org/abs/1706.03762)
- [Building effective agents](https://www.anthropic.com/engineering/building-effective-agents)

Original teaching fixtures, source passages and numerical protocol limits were authored for this class. They are not real hardware evidence. For a project extension, preserve your own edited notebook, code, data, versions and evaluation; do not relabel scaffold results as your own measured experiment.